In [0]:
%run ../helpers/common_utilities

In [0]:
user_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
pipeline_name = "IOT_sensor_data_ingest"

In [0]:
# Create catalog if it does not exist
query = f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}"
print(query)
spark.sql(query)

# Create volume if it does not exist, with a comment
query = f"""CREATE VOLUME IF NOT EXISTS {CATALOG_VOLUME}
COMMENT 'Volume for storing IOT sensor data' """
print(query)
spark.sql(query)

In [0]:
# Create database with external location set to the volume location
volume_location = f"/Volumes/{CATALOG_NAME}/{Default_schema}/{Volume_name}"
query = f"""
CREATE DATABASE IF NOT EXISTS {BRONZE_DATABASE}
COMMENT 'Database for IOT raw sensor data'

"""
print(query)
spark.sql(query)

In [0]:
# Define the pipeline configuration
pipeline_payload = {
    "name": pipeline_name,
    "catalog": CATALOG_NAME,
    "target": Bronze_schema,
    "libraries": [
        {
            "notebook": {
                "path": f"/Workspace/Repos/{user_name}/heavy-electrical-predictive-maintenance/Bronze_iot_creation/Raw_sensor_data_events"
            }
        }
    ],
    "channel": "current",
    "serverless": True
}
print(pipeline_payload)
pipeline_manager = DatabricksPipelineManager(pipeline_name, pipeline_payload)
pipeline_manager.create_pipeline_if_not_exists()